In [220]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [221]:
import datasets
import os
import json
import random
from mgtbench.loading.dataloader import load
from mgtbench.utils import setup_seed

In [222]:
setup_seed(42)

In [223]:
subject = 'Art'
detectLLM = 'Moonshot'
cut_length = 1000
task = 'task3'
match = True
seed = 0

In [224]:
human_data = datasets.load_dataset("/data1/zzy/MGT-human")
subject_human_data = human_data[subject]
if task == 'task1':
    task_path = '/data1/zzy/AIGen-TASK1'
elif task == 'task2' or task == 'task2_gen':
    task_path = '/data1/zzy/AIGen-TASK2'
elif task == 'task3':
    task_path = '/data1/zzy/AIGen-TASK3'
else:
    raise ValueError(f'Unknown task {task}')

# mgt data
files = os.listdir(task_path)
files = [f for f in files if f.endswith('.json')]
files = [f for f in files if (f.split('.')[0].endswith(detectLLM) and f.startswith(subject))]
mgt_data = None
for f in files:
    with open(f'{task_path}/{f}', 'r') as f:
        js = json.load(f)
        if mgt_data is None:
            mgt_data = js
        else:
            mgt_data += js


No config specified, defaulting to: mgt-human/human
Reusing dataset mgt-human (/home/zhiyuan/.cache/huggingface/datasets/mgt-human/human/1.0.0/b0d19c9dd0e4ff915a6b32eb9f08101558980e4e2d1774a7658eeb578b868bb4)
100%|██████████| 16/16 [00:00<00:00, 1210.00it/s]

caculate sample length

In [237]:
ratio = 2
# e.g. 3 means mgt data : human data = 3 : 1
# e.g. 0.5 means mgt data : human data = 1 : 2
human_len = len(subject_human_data)
mgt_len = len(mgt_data)
human_len, mgt_len

(10218, 2543)

In [238]:
if ratio > 1:
    mgt_sample_len = mgt_len
    human_sample_len = int(mgt_len / ratio)
else:
    if not match:
        mgt_sample_len = mgt_len
        human_sample_len = int(mgt_len / ratio)
    else:
        # match, require more human data than mgt data
        mgt_sample_len = mgt_len
        human_sample_len = int(mgt_len / ratio)
        # mgt/human = ratio, mgt + human = mgt + mgt/ratio <= 2 * mgt len, so mgt sample len <= 2 * mgt len / (1 + 1/ratio)
        # mgt_sample_len = int(2 * mgt_len / (1 + 1/ratio))
        # human_sample_len = int(mgt_sample_len / ratio)

mgt_sample_len, human_sample_len

(2543, 1271)

In [239]:
if human_sample_len > human_len:
    raise ValueError(f'Not enough human data, try to sample {human_sample_len} human data, human data length = {human_len}')

In [240]:
# data mix up
all_data = []
if not match:
    subject_human_data = subject_human_data.shuffle(seed)
    # sample human data
    for i in range(human_sample_len):
        all_data.append({'text': subject_human_data[i]['text'], 'label': 0})
    # sample mgt data (always use all mgt data)
    for i in range(mgt_sample_len):
        all_data.append({'text': mgt_data[i]['generated_text'], 'label': 1})

else: # match
    # cannot shuffle human data,
    # because we need to find corresponding human data by id
    random.shuffle(mgt_data) # in place shuffle
    sampled_match_human_indices = set()
    for i in range(mgt_sample_len):
        all_data.append({'text': mgt_data[i]['generated_text'], 'label': 1})
        if i < human_sample_len: # mgt sample > human sample
            # find the corresponding human data
            all_data.append({'text': subject_human_data[mgt_data[i]['id']]['text'], 'label': 0})
            sampled_match_human_indices.add(mgt_data[i]['id']) 

    # sample the rest human data, avoid duplicate
    if human_sample_len > mgt_sample_len: # ratio < 1
        indices = list(range(human_len))
        random.shuffle(indices)
        cnt = human_sample_len - mgt_sample_len
        for id in indices:
            if id not in sampled_match_human_indices:
                all_data.append({'text': subject_human_data[id]['text'], 'label': 0})
                cnt-=1
            if cnt == 0: break


In [241]:
pos = 0
neg = 0
for d in all_data:
    if d['label'] == 1:
        pos += 1
    else:
        neg += 1
pos, neg

(2543, 1271)

### test load function


In [280]:
subject = 'Literature'
detectLLM = 'Moonshot'
cut_length = 1000
task = 'task3'
match = False
seed = 0
ratio = 0.5

data = load(subject, detectLLM, cut_length=cut_length, task=task, match=match, ratio=ratio)

No config specified, defaulting to: mgt-human/human
Reusing dataset mgt-human (/home/zhiyuan/.cache/huggingface/datasets/mgt-human/human/1.0.0/b0d19c9dd0e4ff915a6b32eb9f08101558980e4e2d1774a7658eeb578b868bb4)
100%|██████████| 16/16 [00:00<00:00, 1191.31it/s]


Loading cached shuffled indices for dataset at /home/zhiyuan/.cache/huggingface/datasets/mgt-human/human/1.0.0/b0d19c9dd0e4ff915a6b32eb9f08101558980e4e2d1774a7658eeb578b868bb4/cache-3687ae28eb51f8a3.arrow


loading data with MGT to human ratio: 0.5


In [281]:
pos = 0
neg = 0
for i in range(len(data['train']['label'])):
    if data['train']['label'][i] == 1:
        pos += 1
    else:
        neg += 1

print('training: ')
print('mgt: ', pos)
print('human: ', neg)

pos = 0
neg = 0

for i in range(len(data['test']['label'])):
    if data['test']['label'][i] == 1:
        pos += 1
    else:
        neg += 1
print('testing: ')  
print('mgt: ', pos)
print('human: ', neg)

training: 
mgt:  4470
human:  8872
testing: 
mgt:  1089
human:  2246
